In [1]:
import os, json, time, urllib.request
from pathlib import Path
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
LEIE = DATA_DIR / "raw" / "leie"
WB = LEIE / "wayback"
WB.mkdir(parents=True, exist_ok=True)
for p in sorted(LEIE.rglob("*")):
    if p.is_file():
        print(p.relative_to(LEIE), round(p.stat().st_size / 1e6, 1), "MB")

UPDATED.csv 10.8 MB
UPDATED_live_2026-10-26.csv 10.8 MB
wayback/LEIE_2019_20190110211040.csv 12.8 MB
wayback/LEIE_2020_20191211184149.csv 13.1 MB
wayback/LEIE_2021_20201218080500.csv 13.4 MB
wayback/LEIE_2022_20211218012824.csv 13.5 MB
wayback/LEIE_2023_20230110203218.csv 13.9 MB
wayback/LEIE_2024_20231128115122.csv 10.0 MB
wayback/LEIE_2025_20241224155928.csv 15.1 MB
wayback/LEIE_2026_20260108040703.csv 15.3 MB
wayback/UPDATED_20190110211040.csv 12.8 MB
wayback/UPDATED_20200310231828.csv 13.1 MB
wayback/UPDATED_20210318000849.csv 13.4 MB
wayback/UPDATED_20220311002415.csv 13.6 MB
wayback/UPDATED_20230110203218.csv 13.9 MB


In [2]:
URL = "oig.hhs.gov/exclusions/downloadables/UPDATED.csv"
WINDOWS = {y: (f"{y}01", f"{y}01") for y in range(2013, 2023)}
WINDOWS[2018] = ("201801", "201802")

def cdx(frm, to):
    q = (f"http://web.archive.org/cdx/search/cdx?url={URL}&from={frm}&to={to}"
         "&output=json&filter=statuscode:200")
    with urllib.request.urlopen(q, timeout=60) as r:
        rows = json.load(r)
    return rows[1:]

snaps = {}
for y, (f, t) in WINDOWS.items():
    rows = cdx(f, t)
    snaps[y] = rows[0] if rows else None
    print(y, rows[0][1] if rows else "NONE", f"({len(rows)} captures)")
    time.sleep(1)

2013 NONE (0 captures)
2014 NONE (0 captures)
2015 NONE (0 captures)
2016 NONE (0 captures)
2017 20170125090643 (2 captures)
2018 NONE (0 captures)
2019 20190110211040 (1 captures)
2020 NONE (0 captures)
2021 NONE (0 captures)
2022 NONE (0 captures)


In [3]:
for y, s in snaps.items():
    if s is None:
        continue
    ts, orig = s[1], s[2]
    dest = WB / f"UPDATED_{ts}.csv"
    if dest.exists():
        print(y, "already have", dest.name)
        continue
    with urllib.request.urlopen(f"http://web.archive.org/web/{ts}id_/{orig}", timeout=300) as r:
        dest.write_bytes(r.read())
    print(y, dest.name, round(dest.stat().st_size / 1e6, 1), "MB")
    time.sleep(2)

2017 UPDATED_20170125090643.csv 11.9 MB
2019 already have UPDATED_20190110211040.csv


In [4]:
FRAUD = ["1128a1", "1128a2", "1128a3", "1128b4", "1128b7", "1128c3gi", "1128c3gii"]
summary = []
for y, s in snaps.items():
    if s is None:
        continue
    df = pd.read_csv(WB / f"UPDATED_{s[1]}.csv", dtype=str, encoding="latin-1")
    df.columns = df.columns.str.strip().str.upper()
    t = df["EXCLTYPE"].str.strip().str.lower().str.replace(r"[\s()]", "", regex=True)
    npi = df["NPI"].fillna("").str.strip()
    usable = npi.str.fullmatch(r"\d{10}") & (npi != "0000000000")
    summary.append({"year": y, "snapshot": s[1], "rows": len(df),
                    "max_excldate": df["EXCLDATE"].max(),
                    "fraud_rule_rows": int(t.isin(FRAUD).sum()),
                    "fraud_rule_with_npi": int((t.isin(FRAUD) & usable).sum()),
                    "rule_codes_seen": sorted(set(t) & set(FRAUD))})
print(pd.DataFrame(summary).to_string(index=False))

 year       snapshot  rows max_excldate  fraud_rule_rows  fraud_rule_with_npi                          rule_codes_seen
 2017 20170125090643 66286     20170119            57978                 3212 [1128a1, 1128a2, 1128a3, 1128b4, 1128b7]
 2019 20190110211040 70973     20190120            62192                 4007 [1128a1, 1128a2, 1128a3, 1128b4, 1128b7]


In [5]:
def cdx_raw(params):
    q = "http://web.archive.org/cdx/search/cdx?" + params + "&output=json"
    with urllib.request.urlopen(q, timeout=120) as r:
        rows = json.load(r)
    return pd.DataFrame(rows[1:], columns=rows[0]) if rows else pd.DataFrame()

a = cdx_raw(f"url={URL}&from=2012&to=2019&fl=timestamp,original,statuscode,mimetype,length")
print("A: same file, any status")
print(a.groupby([a["timestamp"].str[:4], "statuscode"]).size() if len(a) else "no captures")

b = cdx_raw("url=oig.hhs.gov/exclusions/&matchType=prefix&from=2012&to=2019"
            "&fl=original,timestamp,statuscode&filter=statuscode:200&collapse=urlkey&limit=2000")
print("\nB: other LEIE-like files under /exclusions/")
print(b[b["original"].str.contains(r"\.(csv|zip|dbf|xls)", case=False)].to_string(index=False)
      if len(b) else "none")

A: same file, any status
timestamp  statuscode
2016       200           2
2017       -             1
           200           9
           302           1
2018       200           1
2019       200           2
dtype: int64

B: other LEIE-like files under /exclusions/
                                                           original      timestamp statuscode
     https://oig.hhs.gov/exclusions/downloadables/2010/rein1009.zip 20140630191725        200
      http://oig.hhs.gov/exclusions/downloadables/2010/rein1010.zip 20120122133452        200
      http://oig.hhs.gov/exclusions/downloadables/2010/rein1011.zip 20120122104243        200
      http://oig.hhs.gov/exclusions/downloadables/2010/rein1012.zip 20120122144811        200
      http://oig.hhs.gov/exclusions/downloadables/2010/sanc1010.zip 20120122140213        200
      http://oig.hhs.gov/exclusions/downloadables/2010/sanc1011.zip 20120122142727        200
      http://oig.hhs.gov/exclusions/downloadables/2010/sanc1012.zip 2012012

/tmp/ipykernel_172942/1717713295.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(b[b["original"].str.contains(r"\.(csv|zip|dbf|xls)", case=False)].to_string(index=False)


In [6]:
full = pd.concat([cdx_raw(f"url={u}&from=2012&to=2019&fl=timestamp,original,statuscode,length")
                  for u in ["oig.hhs.gov/exclusions/downloadables/UPDATED.csv",
                            "oig.hhs.gov/exclusions/downloadables/updatedleie.zip",
                            "oig.hhs.gov/exclusions/downloadables/updatednpiwai.zip"]],
                 ignore_index=True)
print(full.sort_values("timestamp").to_string(index=False))

     timestamp                                                      original statuscode  length
20120123114852   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2163092
20120123235818   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2163089
20120124145704   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2163092
20120509220434   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202976
20120514192027   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202982
20120518100411   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202983
20120519205121   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202981
20120521013641   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202979
20120525070520   http://oig.hhs.gov/exclusions/downloadables/updatedleie.zip        200 2202979
20120528072856   http://oig.hhs.gov/excl

In [7]:
import re, zipfile, io
SUPP = LEIE / "supplements"
SUPP.mkdir(exist_ok=True)

recs = []
for _, r in b.iterrows():
    m1 = re.search(r"/(sanc|rein)(\d{2})(\d{2})(npiwai)?\.(zip|csv)$", r["original"], re.I)
    m2 = re.search(r"/(\d{2})(\d{2})(EXCL|REIN)\.csv$", r["original"], re.I)
    if m1:
        kind = "EXCL" if m1.group(1).lower() == "sanc" else "REIN"
        yy, mm, ext = m1.group(2), m1.group(3), m1.group(5).lower()
    elif m2:
        yy, mm, kind, ext = m2.group(1), m2.group(2), m2.group(3).upper(), "csv"
    else:
        continue
    recs.append({"ym": f"20{yy}{mm}", "kind": kind, "ext": ext, "ts": r["timestamp"], "url": r["original"]})

s = pd.DataFrame(recs)
s = s[(s["ym"] >= "201301") & (s["ym"] <= "201812")]
s = s.sort_values(["ym", "kind", "ext"]).drop_duplicates(["ym", "kind"], keep="first")
print(len(s), "files")
print(s.groupby("kind")["ym"].agg(["count", "min", "max"]))
all_months = set(pd.period_range("2013-01", "2018-12", freq="M").strftime("%Y%m"))
print("Months with no EXCL file:", sorted(all_months - set(s.loc[s["kind"] == "EXCL", "ym"])))
print("Months with no REIN file:", sorted(all_months - set(s.loc[s["kind"] == "REIN", "ym"])))

144 files
      count     min     max
kind                       
EXCL     72  201301  201812
REIN     72  201301  201812
Months with no EXCL file: []
Months with no REIN file: []


In [9]:
def fetch(url, tries=6):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=300) as resp:
                return resp.read()
        except Exception as e:
            wait = 60 * (i + 1)
            print("  retry", i + 1, "in", wait, "s:", str(e)[:60])
            time.sleep(wait)
    raise RuntimeError("gave up: " + url)

for _, r in s.iterrows():
    dest = SUPP / f"{r['ym']}_{r['kind']}.csv"
    if dest.exists() or dest.with_suffix(".zip").exists():
        continue
    data = fetch(f"https://web.archive.org/web/{r['ts']}id_/{r['url']}")
    (dest.with_suffix(".zip") if r["ext"] == "zip" else dest).write_bytes(data)
    print(r["ym"], r["kind"], r["ext"], round(len(data) / 1e3), "KB")
    time.sleep(5)
print("done")

201501 EXCL zip 19 KB
201501 REIN zip 2 KB
201502 EXCL zip 13 KB
201502 REIN zip 3 KB
201503 EXCL zip 15 KB
201503 REIN zip 3 KB
201504 EXCL zip 16 KB
201504 REIN zip 3 KB
201505 EXCL zip 20 KB
201505 REIN zip 3 KB
201506 EXCL zip 16 KB
201506 REIN zip 2 KB
201507 EXCL zip 23 KB
201507 REIN zip 3 KB
201508 EXCL zip 21 KB
201508 REIN zip 4 KB
201509 EXCL zip 13 KB
201509 REIN zip 3 KB
201510 EXCL zip 4 KB
201510 REIN zip 3 KB
201511 EXCL csv 24 KB
201511 REIN csv 9 KB
201512 EXCL csv 65 KB
201512 REIN csv 12 KB
201601 EXCL csv 90 KB
201601 REIN csv 5 KB
201602 EXCL csv 43 KB
201602 REIN csv 10 KB
201603 EXCL csv 74 KB
201603 REIN csv 5 KB
201604 EXCL csv 66 KB
201604 REIN csv 8 KB
201605 EXCL csv 81 KB
201605 REIN csv 9 KB
201606 EXCL csv 66 KB
201606 REIN csv 14 KB
201607 EXCL csv 53 KB
201607 REIN csv 4 KB
201608 EXCL csv 14 KB
201608 REIN csv 3 KB
201609 EXCL csv 86 KB
201609 REIN csv 18 KB
201610 EXCL csv 28 KB
201610 REIN csv 12 KB
201611 EXCL csv 54 KB
201611 REIN csv 7 KB
201612 

In [10]:
v = []
for p in sorted(SUPP.glob("*.csv")):
    try:
        d = pd.read_csv(p, dtype=str, encoding="latin-1")
        d.columns = d.columns.str.strip().str.upper()
        ex = d["EXCLDATE"] if "EXCLDATE" in d else pd.Series(dtype=str)
        v.append({"file": p.name, "rows": len(d), "ncols": d.shape[1], "has_npi": "NPI" in d,
                  "excl_min": ex.min(), "excl_max": ex.max()})
    except Exception as e:
        v.append({"file": p.name, "error": str(e)[:80]})
v = pd.DataFrame(v)
print(v.to_string(index=False))
v.to_csv(DATA_DIR / "processed" / "part_b_1319" / "audit" / "leie_supplements_check.csv", index=False)

           file  rows  ncols  has_npi excl_min excl_max
201511_EXCL.csv   129     18     True 20151025 20151220
201511_REIN.csv    51     18     True 19920803 20150820
201512_EXCL.csv   348     18     True 20150729 20160120
201512_REIN.csv    65     18     True 19890602 20150820
201601_EXCL.csv   484     18     True 20151221 20160218
201601_REIN.csv    29     18     True 19970106 20150120
201602_EXCL.csv   233     18     True 20151029 20160320
201602_REIN.csv    53     18     True 19910905 20151030
201603_EXCL.csv   396     18     True 20160223 20160420
201603_REIN.csv    29     18     True 19830720 20160218
201604_EXCL.csv   359     18     True 20160401 20160519
201604_REIN.csv    44     18     True 19961110 20151020
201605_EXCL.csv   437     18     True 20160420 20160620
201605_REIN.csv    49     18     True 19970907 20160519
201606_EXCL.csv   359     18     True 20160531 20160720
201606_REIN.csv    75     18     True 19860812 20160620
201607_EXCL.csv   284     18     True 20160701 2

In [11]:
import datetime, tempfile
from dbfread import DBF

for zp in sorted(SUPP.glob("*.zip")):
    dest = zp.with_suffix(".csv")
    if dest.exists():
        continue
    z = zipfile.ZipFile(zp)
    names = z.namelist()
    csvs = [n for n in names if n.lower().endswith((".csv", ".txt"))]
    dbfs = [n for n in names if n.lower().endswith(".dbf")]
    if csvs:
        dest.write_bytes(z.read(csvs[0]))
    elif dbfs:
        with tempfile.TemporaryDirectory() as td:
            path = z.extract(dbfs[0], td)
            df = pd.DataFrame(iter(DBF(path, encoding="latin-1", char_decode_errors="replace",
                                       ignore_missing_memofile=True)))
        for c in df.columns:
            if df[c].map(lambda x: isinstance(x, datetime.date)).any():
                df[c] = df[c].map(lambda x: x.strftime("%Y%m%d") if isinstance(x, datetime.date) else x)
        df.to_csv(dest, index=False)
    else:
        print("SKIP", zp.name, names)
        continue
    print(zp.name, "->", dest.name)

201301_EXCL.zip -> 201301_EXCL.csv
201301_REIN.zip -> 201301_REIN.csv
201302_EXCL.zip -> 201302_EXCL.csv
201302_REIN.zip -> 201302_REIN.csv
201303_EXCL.zip -> 201303_EXCL.csv
201303_REIN.zip -> 201303_REIN.csv
201304_EXCL.zip -> 201304_EXCL.csv
201304_REIN.zip -> 201304_REIN.csv
201305_EXCL.zip -> 201305_EXCL.csv
201305_REIN.zip -> 201305_REIN.csv
201306_EXCL.zip -> 201306_EXCL.csv
201306_REIN.zip -> 201306_REIN.csv
201307_EXCL.zip -> 201307_EXCL.csv
201307_REIN.zip -> 201307_REIN.csv
201308_EXCL.zip -> 201308_EXCL.csv
201308_REIN.zip -> 201308_REIN.csv
201309_EXCL.zip -> 201309_EXCL.csv
201309_REIN.zip -> 201309_REIN.csv
201310_EXCL.zip -> 201310_EXCL.csv
201310_REIN.zip -> 201310_REIN.csv
201311_EXCL.zip -> 201311_EXCL.csv
201311_REIN.zip -> 201311_REIN.csv
201312_EXCL.zip -> 201312_EXCL.csv
201312_REIN.zip -> 201312_REIN.csv
201401_EXCL.zip -> 201401_EXCL.csv
201401_REIN.zip -> 201401_REIN.csv
201402_EXCL.zip -> 201402_EXCL.csv
201402_REIN.zip -> 201402_REIN.csv
201403_EXCL.zip -> 2

In [12]:
months = pd.period_range("2013-01", "2018-12", freq="M").strftime("%Y%m")
missing = [f"{m}_{k}" for m in months for k in ["EXCL", "REIN"]
           if not (SUPP / f"{m}_{k}.csv").exists()]
print("Expected 144 | present:", 144 - len(missing), "| missing:", missing or "none")

Expected 144 | present: 144 | missing: none
